In [1]:
%pip install -q python-chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 53.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import glob
import time
import chess
import lightgbm as lgb
from joblib import Parallel, delayed
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

In [3]:
# Veriyi bul ve yükle
yol = glob.glob("/kaggle/input/**/puzzles_500k.parquet", recursive=True)[0]
df = pd.read_parquet(yol)

# Sadece kullanacağımız sütunlar (sızıntı riski olanlar baştan dışarıda)
df = df[["PuzzleId", "FEN", "Moves", "Rating", "GameUrl"]].copy()

# Oyun kimliği: lichess.org/ sonrasındaki 8 karakter
df["oyun_id"] = df["GameUrl"].str.extract(r"lichess\.org/([A-Za-z0-9]{8})")[0]

print("Boyut:", df.shape)
print("Eksik değer:\n", df.isna().sum())
print("\nRating özeti:\n", df["Rating"].describe().round(1))

oyun_basi = df["oyun_id"].value_counts()
print("\nFarklı oyun sayısı:", oyun_basi.size)
print("Birden fazla bulmaca çıkan oyun sayısı:", (oyun_basi > 1).sum())
print("Bir oyundan çıkan en fazla bulmaca:", oyun_basi.max())
df.head(3)

Boyut: (500000, 6)
Eksik değer:
 PuzzleId    0
FEN         0
Moves       0
Rating      0
GameUrl     0
oyun_id     0
dtype: int64

Rating özeti:
 count    500000.0
mean       1500.5
std         475.9
min         399.0
25%        1130.0
50%        1469.0
75%        1854.0
max        2969.0
Name: Rating, dtype: float64

Farklı oyun sayısı: 500000
Birden fazla bulmaca çıkan oyun sayısı: 0
Bir oyundan çıkan en fazla bulmaca: 1


,PuzzleId,FEN,Moves,Rating,GameUrl,oyun_id
0,bkpg3,r3r1k1/2q2pp1/2p4p/1p6/pPpP1nP1/P1P2N1P/7Q/R3R...,f4e2 e1e2 c7h2 e2h2,1425,https://lichess.org/IIRF4olT/black#50,IIRF4olT
1,4AG2R,5rk1/pp3p1p/2p2p1q/4PQ2/4BR2/5KP1/3r3P/8 b - -...,h6h2 f4g4 g8h8 f5f6,1646,https://lichess.org/oCqTk0o6/black#50,oCqTk0o6
2,VTR9q,r3k2r/ppp2pp1/2p5/2P5/4P1b1/2P5/PP3P1p/RNB1R2K...,c1f4 g4f3,695,https://lichess.org/zMIQncOt#31,zMIQncOt


In [4]:
BANT_KENAR = [0, 1000, 1400, 1700, 4000]
BANT_AD = ["<1000", "1000–1399", "1400–1699", "≥1700"]
df["bant"] = pd.cut(df["Rating"], BANT_KENAR, labels=BANT_AD, right=False)

# Önce %70 eğitim / %30 geri kalan, sonra geri kalanı yarıya: %15 doğrulama / %15 test
train, gecici = train_test_split(df, test_size=0.30, stratify=df["bant"], random_state=42)
val, test = train_test_split(gecici, test_size=0.50, stratify=gecici["bant"], random_state=42)

print(f"Eğitim: {len(train):,}  Doğrulama: {len(val):,}  Test: {len(test):,}")

ozet = pd.DataFrame({
    ad: {"Ortalama": p["Rating"].mean(), "Medyan": p["Rating"].median(), "Std": p["Rating"].std(),
         **(p["bant"].value_counts(normalize=True).sort_index() * 100).round(2).to_dict()}
    for ad, p in [("Eğitim", train), ("Doğrulama", val), ("Test", test)]
}).round(1)
ozet

Eğitim: 350,000  Doğrulama: 75,000  Test: 75,000


,Eğitim,Doğrulama,Test
Ortalama,1500.1,1501.5,1501.8
Medyan,1469.0,1469.0,1470.0
Std,475.7,476.0,476.8
<1000,16.4,16.4,16.4
1000–1399,28.8,28.8,28.8
1400–1699,20.8,20.8,20.8
≥1700,34.1,34.1,34.1


In [5]:
SONUCLAR = []

def degerlendir(ad, y_gercek, y_tahmin, bant):
    """Genel ve bant bazında hata ölçülerini hesaplar, sonuç tablosuna ekler."""
    satir = {
        "Model": ad,
        "RMSE": np.sqrt(mean_squared_error(y_gercek, y_tahmin)),
        "MAE": mean_absolute_error(y_gercek, y_tahmin),
        "R²": r2_score(y_gercek, y_tahmin),
    }
    hata = pd.DataFrame({"bant": bant.values, "kare_hata": (np.asarray(y_gercek) - np.asarray(y_tahmin)) ** 2})
    for b, g in hata.groupby("bant", observed=True):
        satir[f"RMSE {b}"] = np.sqrt(g["kare_hata"].mean())
    SONUCLAR.append(satir)
    return pd.DataFrame(SONUCLAR).set_index("Model").round(3)

y_train, y_val = train["Rating"], val["Rating"]

# Temel model 1: her bulmacaya eğitim setinin ortalamasını tahmin et (girdi kullanılmaz)
dummy = DummyRegressor(strategy="mean").fit(np.zeros((len(train), 1)), y_train)
tahmin = dummy.predict(np.zeros((len(val), 1)))

print("Tahmin edilen sabit değer:", round(tahmin[0], 1))
degerlendir("Temel 1: Ortalama", y_val, tahmin, val["bant"])

Tahmin edilen sabit değer: 1500.1


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.0,676.789,318.016,99.617,594.353


In [6]:
# Çözüm uzunluğu: Moves'taki hamle sayısının yarısı (ilk hamle rakibin)
for p in (train, val, test):
    p["cozum_uzunlugu"] = (p["Moves"].str.count(" ") + 1) // 2

print(train["cozum_uzunlugu"].value_counts().sort_index())

agac = DecisionTreeRegressor(max_depth=3, random_state=42)
agac.fit(train[["cozum_uzunlugu"]], y_train)
tahmin = agac.predict(val[["cozum_uzunlugu"]])

degerlendir("Temel 2: Karar ağacı (çözüm uzunluğu)", y_val, tahmin, val["bant"])

cozum_uzunlugu
1      40818
2     182759
3      99351
4      20376
5       4727
6       1324
7        427
8        142
9         40
10        22
11        11
12         2
13         1
Name: count, dtype: int64


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357


In [7]:
def _parca_isle(fonk, fenler, hamleler):
    return [fonk(f, m) for f, m in zip(fenler, hamleler)]

def paralel_cikar(fonk, sutunlar, parca=20_000):
    """fonk(fen, moves) -> tuple; tüm df için paralel hesaplar, df ile aynı indeksli DataFrame döner."""
    t0 = time.time()
    fen, mv = df["FEN"].tolist(), df["Moves"].tolist()
    sonuc = Parallel(n_jobs=-1)(
        delayed(_parca_isle)(fonk, fen[i:i + parca], mv[i:i + parca]) for i in range(0, len(fen), parca))
    ozl = pd.DataFrame([s for grup in sonuc for s in grup], columns=sutunlar, index=df.index)
    print(f"{len(ozl):,} bulmaca, {time.time() - t0:.0f} saniye")
    return ozl

def ilk_hamle_ozellikleri(fen, moves):
    tahta = chess.Board(fen)
    h = moves.split()
    tahta.push_uci(h[0])                         # rakibin hamlesi -> oyuncunun gördüğü pozisyon
    dogru = chess.Move.from_uci(h[1])            # çözümün ilk hamlesi
    return (int(tahta.gives_check(dogru)),       # şah çekiyor mu
            int(tahta.is_capture(dogru)),        # taş alıyor mu
            tahta.piece_type_at(dogru.from_square))  # 1=piyon 2=at 3=fil 4=kale 5=vezir 6=şah

ozl = paralel_cikar(ilk_hamle_ozellikleri, ["ilk_sah", "ilk_alma", "ilk_tas"])
train, val, test = [p.join(ozl) for p in (train, val, test)]

# Hızlı kontrol: eğitim setinde her gruptaki bulmaca sayısı ve medyan puan
for s in ["ilk_sah", "ilk_alma", "ilk_tas"]:
    print(train.groupby(s)["Rating"].agg(["size", "median"]).T, "\n")

500,000 bulmaca, 29 saniye
ilk_sah         0         1
size     140661.0  209339.0
median     1731.0    1266.0 

ilk_alma         0         1
size      209002.0  140998.0
median      1453.0    1490.0 

ilk_tas        1        2        3        4         5        6
size     41396.0  50756.0  40725.0  90778.0  109524.0  16821.0
median    1687.0   1437.0   1520.0   1446.0    1349.0   1793.0 



In [8]:
KATEGORIK = ["ilk_tas"]   # sayı gibi görünen ama sırası anlamsız sütunlar

def lgb_dene(ad, sutunlar, params=None):
    """Verilen sütunlarla LightGBM eğitir, doğrulamada erken durdurur, sonucu tabloya ekler."""
    p = dict(objective="regression", learning_rate=0.05, num_leaves=63, n_estimators=3000,
             min_child_samples=50, random_state=42, verbose=-1)
    p.update(params or {})
    kat = [s for s in sutunlar if s in KATEGORIK]
    Xtr, Xva = train[sutunlar].copy(), val[sutunlar].copy()
    for s in kat:
        Xtr[s], Xva[s] = Xtr[s].astype("category"), Xva[s].astype("category")
    model = lgb.LGBMRegressor(**p)
    model.fit(Xtr, y_train, eval_set=[(Xva, y_val)], categorical_feature=kat or "auto",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    print(f"{ad}: {model.best_iteration_} ağaç")
    return model, degerlendir(ad, y_val, model.predict(Xva, num_iteration=model.best_iteration_), val["bant"])

OZN = ["cozum_uzunlugu", "ilk_sah", "ilk_alma", "ilk_tas"]
model, tablo = lgb_dene("LGBM 1: uzunluk + ilk hamle", OZN)
tablo

LGBM 1: uzunluk + ilk hamle: 96 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808


In [9]:
def cozum_ozellikleri(fen, moves):
    tahta = chess.Board(fen)
    h = moves.split()
    tahta.push_uci(h[0])                              # rakibin ilk hamlesi
    sah = alma = sessiz = terfi = 0
    for i, uci in enumerate(h[1:]):
        m = chess.Move.from_uci(uci)
        if i % 2 == 0:                                # oyuncunun hamleleri (1., 3., 5. ...)
            c, a = tahta.gives_check(m), tahta.is_capture(m)
            sah += c; alma += a; sessiz += (not c and not a)
            terfi += m.promotion is not None
        tahta.push(m)
    n = (len(h)) // 2
    return sah, alma, sessiz, round(sessiz / n, 3), int(terfi > 0), int(tahta.is_checkmate())

YENI = ["cozum_sah", "cozum_alma", "cozum_sessiz", "sessiz_oran", "terfi_var", "mat_bitis"]
ozl = paralel_cikar(cozum_ozellikleri, YENI)
train, val, test = [p.join(ozl) for p in (train, val, test)]

print(train.groupby("mat_bitis")["Rating"].agg(["size", "median"]).T, "\n")
print(train.groupby(train["cozum_sessiz"].clip(upper=3))["Rating"].agg(["size", "median"]).T)

OZN = OZN + YENI
model, tablo = lgb_dene("LGBM 2: + çözüm dizisi", OZN)
tablo

500,000 bulmaca, 47 saniye
mat_bitis         0        1
size       251240.0  98760.0
median       1620.0   1116.0 

cozum_sessiz         0        1        2        3
size          232457.0  77900.0  28672.0  10971.0
median          1292.0   1769.0   1923.0   1987.0
LGBM 2: + çözüm dizisi: 204 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808
LGBM 2: + çözüm dizisi,357.543,288.138,0.436,395.869,315.969,255.845,419.057


In [10]:
DEGER = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}

def materyal(tahta, renk):
    return sum(DEGER[t.piece_type] * (1 if t.color == renk else -1) for t in tahta.piece_map().values())

def materyal_ozellikleri(fen, moves):
    tahta = chess.Board(fen)
    h = moves.split()
    tahta.push_uci(h[0])
    cozen = tahta.turn
    bas = materyal(tahta, cozen)

    # İlk doğru hamle: neyi alıyor, gittiği kare tehlikede mi?
    ilk = chess.Move.from_uci(h[1])
    alinan = tahta.piece_at(ilk.to_square)
    alinan_deger = DEGER[alinan.piece_type] if alinan else (1 if tahta.is_en_passant(ilk) else 0)
    tahta.push(ilk)
    hedef_saldirida = int(tahta.is_attacked_by(not cozen, ilk.to_square))
    hedef_korunuyor = int(tahta.is_attacked_by(cozen, ilk.to_square))
    tahta.pop()

    # Çözüm boyunca materyal dengesi
    en_dusuk = bas
    for uci in h[1:]:
        tahta.push_uci(uci)
        en_dusuk = min(en_dusuk, materyal(tahta, cozen))
    son = materyal(tahta, cozen)
    return bas, son - bas, en_dusuk - bas, int(en_dusuk < bas), alinan_deger, hedef_saldirida, hedef_korunuyor

YENI = ["materyal_bas", "net_kazanc", "en_buyuk_dusus", "fedakarlik", "ilk_alinan_deger", "ilk_hedef_saldirida", "ilk_hedef_korunuyor"]
ozl = paralel_cikar(materyal_ozellikleri, YENI)
train, val, test = [p.join(ozl) for p in (train, val, test)]

print(train.groupby("fedakarlik")["Rating"].agg(["size", "median"]).T, "\n")
print(train.groupby("ilk_hedef_saldirida")["Rating"].agg(["size", "median"]).T)

OZN = OZN + YENI
model, tablo = lgb_dene("LGBM 3: + materyal / fedakârlık", OZN)
tablo

500,000 bulmaca, 92 saniye
fedakarlik         0         1
size        234430.0  115570.0
median        1378.0    1617.0 

ilk_hedef_saldirida         0         1
size                 153821.0  196179.0
median                 1466.0    1471.0
LGBM 3: + materyal / fedakârlık: 1564 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808
LGBM 2: + çözüm dizisi,357.543,288.138,0.436,395.869,315.969,255.845,419.057
LGBM 3: + materyal / fedakârlık,335.933,267.739,0.502,353.414,303.844,256.532,391.215


In [11]:
def pozisyon_ozellikleri(fen, moves):
    tahta = chess.Board(fen)
    h = moves.split()
    tahta.push_uci(h[0])
    cozen, rakip = tahta.turn, not tahta.turn
    dogru = chess.Move.from_uci(h[1])

    def tur(m):  # 0 = sessiz, 1 = alma, 2 = şah
        return 2 if tahta.gives_check(m) else (1 if tahta.is_capture(m) else 0)

    yasal = list(tahta.legal_moves)
    turler = [tur(m) for m in yasal]
    sah_secenegi, alma_secenegi = turler.count(2), turler.count(1)
    aday_kalabaligi = turler.count(tur(dogru))       # doğru hamleyle aynı türden kaç seçenek var

    rakip_asili = cozen_asili = rakip_acmaz = 0
    for kare, tas in tahta.piece_map().items():
        if tas.piece_type == chess.KING:
            continue
        renk = tas.color
        saldirilan = tahta.is_attacked_by(not renk, kare)
        korunan = tahta.is_attacked_by(renk, kare)
        if saldirilan and not korunan:
            if renk == rakip: rakip_asili += 1
            else: cozen_asili += 1
        if renk == rakip and tahta.is_pinned(rakip, kare):
            rakip_acmaz += 1

    rsah = tahta.king(rakip)
    cevre = list(chess.SquareSet(chess.BB_KING_ATTACKS[rsah]))
    cevre_saldiri = sum(tahta.is_attacked_by(cozen, k) for k in cevre)
    kacis = sum(1 for k in cevre if not tahta.is_attacked_by(cozen, k)
                and not (tahta.piece_at(k) and tahta.piece_at(k).color == rakip))

    return (len(yasal), sah_secenegi, alma_secenegi, aday_kalabaligi,
            rakip_asili, cozen_asili, rakip_acmaz, cevre_saldiri, kacis, int(tahta.is_check()))

YENI = ["yasal_hamle", "sah_secenegi", "alma_secenegi", "aday_kalabaligi",
        "rakip_asili", "cozen_asili", "rakip_acmaz", "rakip_sah_cevre_saldiri", "rakip_sah_kacis", "cozen_sah_altinda"]
ozl = paralel_cikar(pozisyon_ozellikleri, YENI)
train, val, test = [p.join(ozl) for p in (train, val, test)]

print(train.groupby(train["aday_kalabaligi"].clip(upper=8))["Rating"].agg(["size", "median"]).T, "\n")
print(train.groupby(train["rakip_sah_kacis"].clip(upper=4))["Rating"].agg(["size", "median"]).T)

OZN = OZN + YENI
model, tablo = lgb_dene("LGBM 4: + pozisyon / taktik", OZN)
tablo

500,000 bulmaca, 160 saniye
aday_kalabaligi        1        2        3        4        5        6       7  \
size             72423.0  80810.0  54128.0  35048.0  21333.0  13633.0  8819.0   
median            1237.0   1316.0   1436.0   1503.0   1542.0   1559.0  1591.0   

aday_kalabaligi        8  
size             63806.0  
median            1797.0   

rakip_sah_kacis        0        1        2        3        4
size             32904.0  88027.0  94672.0  62950.0  71447.0
median            1294.0   1406.0   1465.0   1545.0   1560.0
LGBM 4: + pozisyon / taktik: 2787 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808
LGBM 2: + çözüm dizisi,357.543,288.138,0.436,395.869,315.969,255.845,419.057
LGBM 3: + materyal / fedakârlık,335.933,267.739,0.502,353.414,303.844,256.532,391.215
LGBM 4: + pozisyon / taktik,307.305,241.743,0.583,283.056,285.814,245.406,364.563


In [12]:
model, tablo = lgb_dene("LGBM 4b: ağaç sınırı 10.000", OZN, params={"n_estimators": 10_000})
display(tablo)

onem = pd.Series(model.booster_.feature_importance("gain"), index=OZN)
(onem / onem.sum() * 100).sort_values(ascending=False).round(1).to_frame("Katkı %")

LGBM 4b: ağaç sınırı 10.000: 2787 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808
LGBM 2: + çözüm dizisi,357.543,288.138,0.436,395.869,315.969,255.845,419.057
LGBM 3: + materyal / fedakârlık,335.933,267.739,0.502,353.414,303.844,256.532,391.215
LGBM 4: + pozisyon / taktik,307.305,241.743,0.583,283.056,285.814,245.406,364.563
LGBM 4b: ağaç sınırı 10.000,307.305,241.743,0.583,283.056,285.814,245.406,364.563


,Katkı %
cozum_sessiz,29.3
cozum_uzunlugu,19.4
net_kazanc,6.8
aday_kalabaligi,6.7
mat_bitis,6.5
sah_secenegi,3.9
yasal_hamle,3.6
alma_secenegi,3.3
ilk_tas,3.2
ilk_sah,2.2


In [13]:
def cozum_aday_ozellikleri(fen, moves):
    tahta = chess.Board(fen)
    h = moves.split()
    tahta.push_uci(h[0])
    adaylar, yasallar, rakip_secenek, rakip_zorunlu = [], [], [], 0
    for i, uci in enumerate(h[1:]):
        m = chess.Move.from_uci(uci)
        yasal = list(tahta.legal_moves)
        if i % 2 == 0:                                   # oyuncunun sırası
            def tur(x):
                return 2 if tahta.gives_check(x) else (1 if tahta.is_capture(x) else 0)
            t = tur(m)
            adaylar.append(sum(1 for x in yasal if tur(x) == t))
            yasallar.append(len(yasal))
        else:                                            # rakibin sırası
            rakip_secenek.append(len(yasal))
            rakip_zorunlu += len(yasal) == 1
        tahta.push(m)
    return (sum(adaylar), max(adaylar), round(np.mean(adaylar), 2), round(np.mean(yasallar), 2),
            round(np.mean(rakip_secenek), 2) if rakip_secenek else 0, rakip_zorunlu)

YENI = ["aday_toplam", "aday_max", "aday_ort", "yasal_ort", "rakip_secenek_ort", "rakip_zorunlu"]
ozl = paralel_cikar(cozum_aday_ozellikleri, YENI)
train, val, test = [p.join(ozl) for p in (train, val, test)]

print(train.groupby(pd.qcut(train["aday_toplam"], 6, duplicates="drop"), observed=True)["Rating"].agg(["size", "median"]).T, "\n")
print(train.groupby(train["rakip_zorunlu"].clip(upper=3))["Rating"].agg(["size", "median"]).T)

OZN = OZN + YENI
model, tablo = lgb_dene("LGBM 5: + çözüm boyunca aday kalabalığı", OZN, params={"n_estimators": 10_000})
tablo

500,000 bulmaca, 285 saniye
aday_toplam  (0.999, 3.0]  (3.0, 5.0]  (5.0, 7.0]  (7.0, 12.0]  (12.0, 30.0]  \
size              64465.0     64683.0     49263.0      59459.0       55363.0   
median             1002.0      1267.0      1407.0       1565.0        1774.0   

aday_toplam  (30.0, 206.0]  
size               56767.0  
median              1907.0   

rakip_zorunlu         0        1        2       3
size           274019.0  64346.0  10778.0   857.0
median           1510.0   1312.0   1447.0  1761.0
LGBM 5: + çözüm boyunca aday kalabalığı: 2785 ağaç


,RMSE,MAE,R²,RMSE <1000,RMSE 1000–1399,RMSE 1400–1699,RMSE ≥1700
Model,,,,,,,
Temel 1: Ortalama,475.982,396.893,-0.000,676.789,318.016,99.617,594.353
Temel 2: Karar ağacı (çözüm uzunluğu),410.871,336.168,0.255,514.931,335.074,209.588,496.357
LGBM 1: uzunluk + ilk hamle,380.642,307.542,0.360,444.868,320.497,241.500,455.808
LGBM 2: + çözüm dizisi,357.543,288.138,0.436,395.869,315.969,255.845,419.057
LGBM 3: + materyal / fedakârlık,335.933,267.739,0.502,353.414,303.844,256.532,391.215
LGBM 4: + pozisyon / taktik,307.305,241.743,0.583,283.056,285.814,245.406,364.563
LGBM 4b: ağaç sınırı 10.000,307.305,241.743,0.583,283.056,285.814,245.406,364.563
LGBM 5: + çözüm boyunca aday kalabalığı,301.445,237.104,0.599,273.340,279.100,245.415,358.100
